# Multi Model Single-Image Enhancement Pipeline (Refactored)
This notebook processes one image at a time and is designed for Colab, Kaggle, and local Jupyter environments.

## Refactor Goals
- Cleaner cell responsibilities
- Consistent and structured logging
- Easier future model additions via config + registry patterns
- Safer execution with explicit validation and result objects

## How To Use This Notebook
1. Run cells from top to bottom once.
2. In the configuration cell, set your input mode and model choices.
3. Run the input cell to resolve your image path.
4. Run inference, then visualization, then validation.
5. Re-run only the configuration + input + inference cells when trying new settings.

## Execution Flow
1. Shared utilities and runtime detection
2. Dependency setup
3. Pretrained model download manager
4. User configuration (typed model + pipeline config)
5. Optional checkpoint compatibility diagnostics
6. Input image resolution
7. Modular inference pipeline (denoise + enhancement)
8. Visualization
9. Validation and smoke checks

In [ ]:
import logging
import os
import subprocess
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional


def setup_logger(name: str = 'image_enhancement_pipeline', level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter('[%(asctime)s] [%(levelname)s] [%(name)s] %(message)s', '%H:%M:%S')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


LOGGER = setup_logger()


def log_section(title: str) -> None:
    LOGGER.info('=' * 88)
    LOGGER.info(title)
    LOGGER.info('=' * 88)


def run_command(cmd: List[str], description: Optional[str] = None) -> None:
    start = time.perf_counter()
    label = description or cmd[0]
    LOGGER.info('[CMD][START] %s', label)
    LOGGER.info('[CMD][RUN] %s', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    elapsed = time.perf_counter() - start
    LOGGER.info('[CMD][DONE] %s (%.2fs)', label, elapsed)


@dataclass(frozen=True)
class RuntimeContext:
    runtime_name: str
    base_dir: Path
    repo_dir: Path


def detect_runtime_context() -> RuntimeContext:
    if Path('/content').exists():
        base_dir = Path('/content')
        runtime_name = 'colab'
    elif Path('/kaggle/working').exists():
        base_dir = Path('/kaggle/working')
        runtime_name = 'kaggle'
    else:
        base_dir = Path.cwd()
        runtime_name = 'local'

    repo_dir = base_dir / 'BasicSR'
    return RuntimeContext(runtime_name=runtime_name, base_dir=base_dir, repo_dir=repo_dir)


log_section('Step 1/9: Detect runtime and prepare BasicSR repository')
RUNTIME_CONTEXT = detect_runtime_context()
LOGGER.info('Runtime detected: %s', RUNTIME_CONTEXT.runtime_name)
LOGGER.info('Base directory: %s', RUNTIME_CONTEXT.base_dir)


if RUNTIME_CONTEXT.repo_dir.exists():
    LOGGER.info('Reusing existing repository at: %s', RUNTIME_CONTEXT.repo_dir)
else:
    run_command(
        ['git', 'clone', 'https://github.com/XPixelGroup/BasicSR.git', str(RUNTIME_CONTEXT.repo_dir)],
        description='Cloning BasicSR repository'
    )


os.chdir(RUNTIME_CONTEXT.repo_dir)
LOGGER.info('Working directory set to: %s', Path.cwd())

In [ ]:
import torch


log_section('Step 2/9: Install dependencies')


run_command([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'], 'Installing torch + torchvision')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], 'Installing BasicSR requirements')
run_command([sys.executable, 'setup.py', 'develop'], 'Installing BasicSR in develop mode')


LOGGER.info('Torch Version: %s', torch.__version__)
LOGGER.info('CUDA Version: %s', torch.version.cuda)
LOGGER.info('CUDNN Version: %s', torch.backends.cudnn.version())
LOGGER.info('CUDA Available: %s', torch.cuda.is_available())

In [ ]:
import urllib.request
from pathlib import Path


log_section('Step 3/9: Download pretrained models')


MODEL_DOWNLOAD_REGISTRY = {
    'RealESRGAN_x4plus': {
        'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        'path': 'experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth'
    },
    'RealESRGAN_general_x4v3': {
        'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
        'path': 'experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth'
    },
    'SwinIR_color_dn_noise15': {
        'url': 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise15.pth',
        'path': 'experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise15.pth'
    },
    'SwinIR_color_dn_noise25': {
        'url': 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise25.pth',
        'path': 'experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise25.pth'
    },
    'SwinIR_color_dn_noise50': {
        'url': 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise50.pth',
        'path': 'experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise50.pth'
    }
}


def download_if_missing(url: str, destination: Path, label: str, retries: int = 2) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        LOGGER.info('[DOWNLOAD][SKIP] %s already exists -> %s', label, destination)
        return

    last_error = None
    for attempt in range(1, retries + 2):
        LOGGER.info('[DOWNLOAD][START] %s attempt=%s url=%s', label, attempt, url)
        try:
            urllib.request.urlretrieve(url, str(destination))
            LOGGER.info('[DOWNLOAD][DONE] %s -> %s', label, destination)
            return
        except Exception as exc:
            last_error = exc
            LOGGER.warning('[DOWNLOAD][RETRY] %s failed on attempt %s: %s', label, attempt, exc)

    raise RuntimeError(f'Failed to download {label} after {retries + 1} attempts: {last_error}')


# ESRGAN weights via BasicSR helper script.
run_command(
    [sys.executable, 'scripts/download_pretrained_models.py', 'ESRGAN'],
    'Downloading ESRGAN model via BasicSR script'
 )


for model_name, meta in MODEL_DOWNLOAD_REGISTRY.items():
    download_if_missing(meta['url'], Path(meta['path']), model_name)


# Optional RIDNet weights.
RIDNET_MODEL_URL = 'https://drive.usercontent.google.com/download?id=1N_eMh58R7qraqtexjCBp9at_I2RhDbcH&export=download'
ridnet_path = Path('experiments/pretrained_models/RIDNet/RIDNet.pth')
ridnet_path.parent.mkdir(parents=True, exist_ok=True)


if ridnet_path.exists():
    LOGGER.info('[DOWNLOAD][SKIP] RIDNet already exists -> %s', ridnet_path)
elif RIDNET_MODEL_URL:
    download_if_missing(RIDNET_MODEL_URL, ridnet_path, 'RIDNet')
else:
    LOGGER.warning(
        'RIDNet weight file not found. Place RIDNet.pth at experiments/pretrained_models/RIDNet/RIDNet.pth to enable RIDNet denoising.'
    )

## Input Mode
- Set USE_UPLOAD = True to upload one image from Colab file picker.
- Set USE_UPLOAD = False to use a direct image path (recommended for Kaggle or local).

## Configuration Notes
- Use AVAILABLE_DENOISE_CONFIGS and AVAILABLE_MODEL_CONFIGS as the single source of truth.
- Add future models by inserting one new ModelConfig entry and extending the registry builder in the inference cell.

## What You Usually Edit In This Notebook
- PIPELINE_CONFIG.use_upload
- PIPELINE_CONFIG.input_image_path
- PIPELINE_CONFIG.enable_denoising
- PIPELINE_CONFIG.denoise_model_name
- PIPELINE_CONFIG.selected_models

In [ ]:
log_section('Step 4/9: Configure input, denoising, and model selection')


@dataclass(frozen=True)
class ModelConfig:
    name: str
    arch: str
    model_path: str
    state_key_priority: List[str]
    params: Dict[str, Any] = field(default_factory=dict)


@dataclass
class PipelineConfig:
    use_upload: bool = True
    input_image_path: str = ''
    output_dir: str = 'results/compare'
    enable_denoising: bool = True
    denoise_model_name: str = 'SwinIR_color_dn_noise25'
    ridnet_allow_fallback: bool = True
    ridnet_fallback_model_name: str = 'SwinIR_color_dn_noise25'
    selected_models: List[str] = field(default_factory=lambda: ['ESRGAN', 'RealESRGAN_x4plus', 'RealESRGAN_general_x4v3'])


AVAILABLE_DENOISE_CONFIGS: Dict[str, ModelConfig] = {
    'SwinIR_color_dn_noise15': ModelConfig(
        name='SwinIR_color_dn_noise15',
        arch='swinir_color_dn',
        model_path='experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise15.pth',
        state_key_priority=['params_ema', 'params']
    ),
    'SwinIR_color_dn_noise25': ModelConfig(
        name='SwinIR_color_dn_noise25',
        arch='swinir_color_dn',
        model_path='experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise25.pth',
        state_key_priority=['params_ema', 'params']
    ),
    'SwinIR_color_dn_noise50': ModelConfig(
        name='SwinIR_color_dn_noise50',
        arch='swinir_color_dn',
        model_path='experiments/pretrained_models/SwinIR/005_colorDN_DFWB_s128w8_SwinIR-M_noise50.pth',
        state_key_priority=['params_ema', 'params']
    ),
    'RIDNet': ModelConfig(
        name='RIDNet',
        arch='ridnet',
        model_path='experiments/pretrained_models/RIDNet/RIDNet.pth',
        state_key_priority=['params_ema', 'params']
    )
}


AVAILABLE_MODEL_CONFIGS: Dict[str, ModelConfig] = {
    'ESRGAN': ModelConfig(
        name='ESRGAN',
        arch='rrdb',
        model_path='experiments/pretrained_models/ESRGAN/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth',
        state_key_priority=['params', 'params_ema'],
        params={'num_block': 23, 'num_grow_ch': 32}
    ),
    'RealESRGAN_x4plus': ModelConfig(
        name='RealESRGAN_x4plus',
        arch='rrdb',
        model_path='experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth',
        state_key_priority=['params_ema', 'params'],
        params={'num_block': 23, 'num_grow_ch': 32}
    ),
    'RealESRGAN_general_x4v3': ModelConfig(
        name='RealESRGAN_general_x4v3',
        arch='srvgg',
        model_path='experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth',
        state_key_priority=['params_ema', 'params'],
        params={'num_conv': 32, 'upscale': 4, 'act_type': 'prelu'}
    )
}


PIPELINE_CONFIG = PipelineConfig()


def validate_pipeline_config(cfg: PipelineConfig) -> None:
    if cfg.enable_denoising and cfg.denoise_model_name not in AVAILABLE_DENOISE_CONFIGS:
        raise ValueError(f'Unknown denoise model: {cfg.denoise_model_name}')

    if cfg.ridnet_allow_fallback:
        if cfg.ridnet_fallback_model_name not in AVAILABLE_DENOISE_CONFIGS:
            raise ValueError(f'Unknown RIDNet fallback model: {cfg.ridnet_fallback_model_name}')
        if cfg.ridnet_fallback_model_name == 'RIDNet':
            raise ValueError('RIDNet fallback model cannot be RIDNet.')

    invalid_models = [m for m in cfg.selected_models if m not in AVAILABLE_MODEL_CONFIGS]
    if invalid_models:
        raise ValueError(f'Unknown enhancement model(s): {invalid_models}')


validate_pipeline_config(PIPELINE_CONFIG)


LOGGER.info('use_upload=%s', PIPELINE_CONFIG.use_upload)
LOGGER.info('output_dir=%s', PIPELINE_CONFIG.output_dir)
LOGGER.info('enable_denoising=%s', PIPELINE_CONFIG.enable_denoising)
LOGGER.info('denoise_model_name=%s', PIPELINE_CONFIG.denoise_model_name)
LOGGER.info('ridnet_allow_fallback=%s', PIPELINE_CONFIG.ridnet_allow_fallback)
LOGGER.info('ridnet_fallback_model_name=%s', PIPELINE_CONFIG.ridnet_fallback_model_name)
LOGGER.info('selected_models=%s', PIPELINE_CONFIG.selected_models)

In [ ]:
import torch


log_section('Step 5/9: Optional checkpoint compatibility diagnostics')


def resolve_checkpoint_state(raw_state: Any, state_key_priority: List[str]) -> tuple[Any, str]:
    if isinstance(raw_state, dict):
        for key in state_key_priority:
            if key in raw_state and isinstance(raw_state[key], dict):
                return raw_state[key], key
    return raw_state, 'root'


def inspect_model_checkpoint(model_cfg: ModelConfig) -> None:
    model_path = Path(model_cfg.model_path)
    LOGGER.info('[CHECKPOINT] Inspecting %s at %s', model_cfg.name, model_path)

    if not model_path.exists():
        LOGGER.warning('[CHECKPOINT] File does not exist: %s', model_path)
        return

    raw_state = torch.load(str(model_path), map_location='cpu')
    state_dict, source_key = resolve_checkpoint_state(raw_state, model_cfg.state_key_priority)

    if not isinstance(state_dict, dict):
        LOGGER.warning('[CHECKPOINT] %s does not contain a parameter dictionary.', model_cfg.name)
        return

    keys = list(state_dict.keys())
    LOGGER.info('[CHECKPOINT] source_key=%s key_count=%s', source_key, len(keys))
    LOGGER.info('[CHECKPOINT] first_keys=%s', keys[:10])

    if model_cfg.name == 'RIDNet':
        looks_like_basicsr_ridnet = any(k.startswith('head.') for k in keys) or any(k.startswith('body.0.') for k in keys)
        looks_like_external_ridnet = any(k.startswith('b1.') for k in keys) or any(k.startswith('head.body.') for k in keys)

        if looks_like_basicsr_ridnet:
            LOGGER.info('[CHECKPOINT] RIDNet checkpoint format appears BasicSR compatible.')
        elif looks_like_external_ridnet:
            LOGGER.warning('[CHECKPOINT] RIDNet checkpoint appears from another implementation. Fallback may be required.')
        else:
            LOGGER.warning('[CHECKPOINT] RIDNet checkpoint format is unknown.')


if PIPELINE_CONFIG.enable_denoising:
    inspect_model_checkpoint(AVAILABLE_DENOISE_CONFIGS[PIPELINE_CONFIG.denoise_model_name])
else:
    LOGGER.info('Denoising disabled. Skipping checkpoint diagnostics.')

In [ ]:
from pathlib import Path


log_section('Step 6/9: Resolve input image path')


class InputManager:
    def __init__(self, upload_dir: str = 'datasets/upload') -> None:
        self.upload_dir = Path(upload_dir)
        self.upload_dir.mkdir(parents=True, exist_ok=True)

    def resolve_input_path(self, use_upload: bool, direct_path: str) -> str:
        if use_upload:
            return self._handle_colab_upload()

        if not direct_path:
            raise ValueError('Set PIPELINE_CONFIG.input_image_path when use_upload is False.')

        path = Path(direct_path)
        if not path.exists():
            raise FileNotFoundError(f'Direct input path not found: {path}')

        return str(path)

    def _handle_colab_upload(self) -> str:
        LOGGER.info('[INPUT] Upload mode enabled. Attempting Colab upload flow.')
        try:
            from google.colab import files
        except Exception as exc:
            raise RuntimeError(
                'Upload mode is only available in Colab runtime. Set use_upload=False and set a direct path.'
            ) from exc

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')

        first_name = next(iter(uploaded))
        dst = self.upload_dir / first_name
        with open(dst, 'wb') as file_handle:
            file_handle.write(uploaded[first_name])

        LOGGER.info('[INPUT] Uploaded image saved at: %s', dst)
        return str(dst)


INPUT_MANAGER = InputManager()
RESOLVED_INPUT_PATH = INPUT_MANAGER.resolve_input_path(
    use_upload=PIPELINE_CONFIG.use_upload,
    direct_path=PIPELINE_CONFIG.input_image_path
)
LOGGER.info('[INPUT] Resolved input image path: %s', RESOLVED_INPUT_PATH)

### Inference Stage
This next cell runs the full modular pipeline.

What happens:
- Loads the input image
- Optionally denoises it
- Runs all selected enhancement models
- Saves outputs and returns a single PIPELINE_RESULT object

Tip: when tuning model selections, rerun this cell after updating config/input cells.

In [ ]:
from datetime import datetime
from pathlib import Path


import cv2
import numpy as np
import torch
from torch.nn import functional as F
from basicsr.archs.ridnet_arch import RIDNet
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.archs.srvgg_arch import SRVGGNetCompact
from basicsr.archs.swinir_arch import SwinIR


log_section('Step 7/9: Run modular denoising and enhancement pipeline')


class ImageProcessor:
    @staticmethod
    def load_bgr(path: str) -> np.ndarray:
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'Failed to read image: {path}')
        return image

    @staticmethod
    def bgr_to_tensor(image_bgr: np.ndarray, device: torch.device) -> torch.Tensor:
        image = image_bgr.astype(np.float32) / 255.0
        image = torch.from_numpy(np.transpose(image[:, :, [2, 1, 0]], (2, 0, 1))).float()
        return image.unsqueeze(0).to(device)

    @staticmethod
    def tensor_to_bgr_uint8(tensor: torch.Tensor) -> np.ndarray:
        output = tensor.data.squeeze().float().cpu().clamp_(0, 1).numpy()
        output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0))
        return (output * 255.0).round().astype(np.uint8)

    @staticmethod
    def save_bgr(image_bgr: np.ndarray, output_path: Path) -> str:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), image_bgr)
        return str(output_path)


class ModelRegistry:
    @staticmethod
    def build_model(model_cfg: ModelConfig) -> torch.nn.Module:
        if model_cfg.arch == 'rrdb':
            return RRDBNet(
                num_in_ch=3,
                num_out_ch=3,
                num_feat=64,
                num_block=model_cfg.params['num_block'],
                num_grow_ch=model_cfg.params['num_grow_ch']
            )
        if model_cfg.arch == 'srvgg':
            return SRVGGNetCompact(
                num_in_ch=3,
                num_out_ch=3,
                num_feat=64,
                num_conv=model_cfg.params['num_conv'],
                upscale=model_cfg.params['upscale'],
                act_type=model_cfg.params['act_type']
            )
        if model_cfg.arch == 'swinir_color_dn':
            return SwinIR(
                upscale=1,
                in_chans=3,
                img_size=128,
                window_size=8,
                img_range=1.0,
                depths=[6, 6, 6, 6, 6, 6],
                embed_dim=180,
                num_heads=[6, 6, 6, 6, 6, 6],
                mlp_ratio=2,
                upsampler='',
                resi_connection='1conv'
            )
        if model_cfg.arch == 'ridnet':
            return RIDNet(3, 64, 3)
        raise ValueError(f'Unsupported model arch: {model_cfg.arch}')

    @staticmethod
    def load_weights(model: torch.nn.Module, model_cfg: ModelConfig, device: torch.device) -> torch.nn.Module:
        model_path = Path(model_cfg.model_path)
        if not model_path.exists():
            raise FileNotFoundError(f'Model weights not found: {model_path}')

        raw_state = torch.load(str(model_path), map_location=device)
        state_dict, state_key = resolve_checkpoint_state(raw_state, model_cfg.state_key_priority)
        LOGGER.info('[WEIGHTS] model=%s state_source=%s path=%s', model_cfg.name, state_key, model_path)
        model.load_state_dict(state_dict, strict=True)
        model.eval()
        return model.to(device)


class InferenceEngine:
    def __init__(self, device: torch.device) -> None:
        self.device = device

    def run_swinir_denoise(self, model: torch.nn.Module, image_tensor: torch.Tensor) -> torch.Tensor:
        window_size = 8
        _, _, height, width = image_tensor.size()
        pad_h = (window_size - height % window_size) % window_size
        pad_w = (window_size - width % window_size) % window_size

        with torch.no_grad():
            padded = F.pad(image_tensor, (0, pad_w, 0, pad_h), 'reflect')
            output = model(padded)
            output = output[:, :, :height, :width]

        return output

    def run_standard(self, model: torch.nn.Module, image_tensor: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return model(image_tensor)


class EnhancementPipeline:
    def __init__(self, pipeline_cfg: PipelineConfig, input_path: str) -> None:
        self.cfg = pipeline_cfg
        self.input_path = Path(input_path)
        self.output_dir = Path(self.cfg.output_dir)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.processor = ImageProcessor()
        self.engine = InferenceEngine(self.device)

    def _run_denoise(self, input_bgr: np.ndarray) -> tuple[np.ndarray, Optional[str], str]:
        selected_name = self.cfg.denoise_model_name
        selected_cfg = AVAILABLE_DENOISE_CONFIGS[selected_name]
        image_tensor = self.processor.bgr_to_tensor(input_bgr, self.device)

        try:
            model = ModelRegistry.build_model(selected_cfg)
            model = ModelRegistry.load_weights(model, selected_cfg, self.device)
            if selected_cfg.arch == 'swinir_color_dn':
                denoised_tensor = self.engine.run_swinir_denoise(model, image_tensor)
            else:
                denoised_tensor = self.engine.run_standard(model, image_tensor)
            used_name = selected_name
        except RuntimeError as exc:
            if not (selected_cfg.arch == 'ridnet' and self.cfg.ridnet_allow_fallback):
                raise
            fallback_name = self.cfg.ridnet_fallback_model_name
            fallback_cfg = AVAILABLE_DENOISE_CONFIGS[fallback_name]
            LOGGER.warning('[DENOISE] RIDNet failed: %s', exc)
            LOGGER.warning('[DENOISE] Falling back to %s', fallback_name)
            model = ModelRegistry.build_model(fallback_cfg)
            model = ModelRegistry.load_weights(model, fallback_cfg, self.device)
            denoised_tensor = self.engine.run_swinir_denoise(model, image_tensor)
            used_name = fallback_name

        denoised_bgr = self.processor.tensor_to_bgr_uint8(denoised_tensor)
        denoised_path = self.output_dir / f'{self.input_path.stem}_{used_name}_denoised.png'
        saved_path = self.processor.save_bgr(denoised_bgr, denoised_path)
        LOGGER.info('[DENOISE] Saved denoised image -> %s', saved_path)
        return denoised_bgr, saved_path, used_name

    def run(self) -> Dict[str, Any]:
        if not self.input_path.exists():
            raise FileNotFoundError(f'Input image not found: {self.input_path}')

        self.output_dir.mkdir(parents=True, exist_ok=True)
        LOGGER.info('[PIPELINE] device=%s', self.device)
        LOGGER.info('[PIPELINE] input=%s', self.input_path)
        LOGGER.info('[PIPELINE] output_dir=%s', self.output_dir)

        original_bgr = self.processor.load_bgr(str(self.input_path))
        inference_bgr = original_bgr
        denoised_path = None
        denoise_model_used = None

        if self.cfg.enable_denoising:
            inference_bgr, denoised_path, denoise_model_used = self._run_denoise(original_bgr)
        else:
            LOGGER.info('[DENOISE] Disabled. Using original input.')

        inference_tensor = self.processor.bgr_to_tensor(inference_bgr, self.device)
        output_paths: Dict[str, str] = {}

        for model_name in self.cfg.selected_models:
            model_cfg = AVAILABLE_MODEL_CONFIGS[model_name]
            LOGGER.info('[ENHANCE][START] model=%s', model_name)
            model = ModelRegistry.build_model(model_cfg)
            model = ModelRegistry.load_weights(model, model_cfg, self.device)
            output_tensor = self.engine.run_standard(model, inference_tensor)
            output_bgr = self.processor.tensor_to_bgr_uint8(output_tensor)
            output_path = self.output_dir / f'{self.input_path.stem}_{model_name}.png'
            output_paths[model_name] = self.processor.save_bgr(output_bgr, output_path)
            LOGGER.info('[ENHANCE][DONE] model=%s path=%s', model_name, output_paths[model_name])

        return {
            'input_path': str(self.input_path),
            'denoised_path': denoised_path,
            'denoise_model_used': denoise_model_used,
            'enhancements': output_paths,
            'metadata': {
                'timestamp': datetime.now().isoformat(timespec='seconds'),
                'device': str(self.device),
                'selected_models': list(self.cfg.selected_models),
                'denoising_enabled': bool(self.cfg.enable_denoising)
            }
        }


PIPELINE = EnhancementPipeline(PIPELINE_CONFIG, RESOLVED_INPUT_PATH)
PIPELINE_RESULT = PIPELINE.run()


# Backward compatibility for downstream cells that still read legacy names.
OUTPUT_PATHS = PIPELINE_RESULT['enhancements']
LAST_INPUT_PATH = PIPELINE_RESULT['input_path']
LAST_DENOISED_PATH = PIPELINE_RESULT['denoised_path']
DENOISE_APPLIED = PIPELINE_RESULT['denoised_path'] is not None


LOGGER.info('Inference complete for %s model(s).', len(OUTPUT_PATHS))
LOGGER.info('Pipeline metadata: %s', PIPELINE_RESULT['metadata'])

### Visualization Stage
This next cell displays input, optional denoised image, and each model output side by side.

Use this to quickly compare visual quality before and after changing models or denoise settings.

In [ ]:
import cv2
import matplotlib.pyplot as plt


log_section('Step 8/9: Visualize and review outputs')


def read_rgb(path: str):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Cannot read image: {path}')
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


if 'PIPELINE_RESULT' not in globals():
    raise ValueError('No pipeline result found. Run the inference cell first.')


input_rgb = read_rgb(PIPELINE_RESULT['input_path'])
model_names = list(PIPELINE_RESULT['enhancements'].keys())
has_denoised = bool(PIPELINE_RESULT.get('denoised_path'))


n_cols = 1 + (1 if has_denoised else 0) + len(model_names)
figure = plt.figure(figsize=(6 * n_cols, 6))


plot_index = 1
ax = figure.add_subplot(1, n_cols, plot_index)
ax.set_title('Input image')
ax.axis('off')
ax.imshow(input_rgb)
plot_index += 1


if has_denoised:
    ax = figure.add_subplot(1, n_cols, plot_index)
    denoise_model_used = PIPELINE_RESULT.get('denoise_model_used') or 'Denoised'
    ax.set_title(f'Denoised ({denoise_model_used})')
    ax.axis('off')
    ax.imshow(read_rgb(PIPELINE_RESULT['denoised_path']))
    plot_index += 1


for model_name in model_names:
    ax = figure.add_subplot(1, n_cols, plot_index)
    suffix = ' (from denoised input)' if has_denoised else ''
    ax.set_title(f'{model_name}{suffix}')
    ax.axis('off')
    ax.imshow(read_rgb(PIPELINE_RESULT['enhancements'][model_name]))
    plot_index += 1


plt.tight_layout()
plt.show()


LOGGER.info('Saved outputs summary:')
LOGGER.info('- Input path: %s', PIPELINE_RESULT['input_path'])
if has_denoised:
    LOGGER.info('- Denoised path: %s', PIPELINE_RESULT['denoised_path'])
for model_name, output_path in PIPELINE_RESULT['enhancements'].items():
    LOGGER.info('- %s: %s', model_name, output_path)

### Validation Stage
This final cell runs simple smoke checks on the generated results.

It verifies expected keys and output files so you can trust the run before sharing or batch-testing more images.

In [ ]:
from pathlib import Path
from typing import Any, Dict


log_section('Step 9/9: Validation and smoke checks')


def validate_pipeline_result(result: Dict[str, Any]) -> None:
    required_keys = ['input_path', 'enhancements', 'metadata']
    missing = [key for key in required_keys if key not in result]
    if missing:
        raise ValueError(f'Missing result keys: {missing}')

    input_path = Path(result['input_path'])
    if not input_path.exists():
        raise FileNotFoundError(f'Input path does not exist: {input_path}')

    denoised_path = result.get('denoised_path')
    if denoised_path and not Path(denoised_path).exists():
        raise FileNotFoundError(f'Denoised output missing: {denoised_path}')

    enhancements = result['enhancements']
    if not enhancements:
        raise ValueError('No enhancement outputs were generated.')

    for model_name, output_path in enhancements.items():
        if not Path(output_path).exists():
            raise FileNotFoundError(f'Output missing for {model_name}: {output_path}')

    LOGGER.info('[VALIDATION] Output files are present for all expected artifacts.')


validate_pipeline_result(PIPELINE_RESULT)
LOGGER.info('[VALIDATION] metadata=%s', PIPELINE_RESULT['metadata'])
LOGGER.info('[VALIDATION] Smoke checks passed.')